# All logic to embbeding movie

#### Variables a conciderar
- adult (2)
- original_lenguaje (4)
- title (3)
- keywords (5)
- popularity (3)
- date (year) (3)
- director (5)
- actors (5)
- vote_average (3)
- vote_count (3)
- Genre (7)
-------------------------
43

## Connect to DB

In [1]:
import mysql.connector
import pandas as pd
#-----------------------------
import random
import mmh3
import numpy as np
#-----------------------------
import gensim.downloader as api
import ast
#-----------------------------
from annoy import AnnoyIndex

In [2]:
config = {
    'host': '127.0.0.1',
    'port':'3307',
    'user': 'REG',
    'password': 'Aa123456',
    'database': 'TV_MOVIES_DB'
}


In [3]:
conexion = mysql.connector.connect(**config)
cursor = conexion.cursor()
consulta = "SELECT * FROM Movies"
cursor.execute(consulta)

results = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

movie_df = pd.DataFrame(results, columns=column_names)

In [4]:
movie_df.head(2)

,id,id_tmdb,adult,backdrop_path,original_lenguaje,overview,keywords,popularity,poster_path,release_date,title,director,actors,vote_average,vote_count,image_path,created_at
0,1,522627,0,/tintsaQ0WLzZsTMkTiqtMB3rfc8.jpg,en,American expat Mickey Pearson has built a high...,"['profitable marijuana', 'expat mickey', 'lond...",8.102,/jtrhTYB7xSrJxR1vusu99nvnZ1g.jpg,2020-01-01,The Gentlemen,Guy Ritchie,"['Matthew McConaughey', 'Charlie Hunnam', 'Mic...",7.670,6379,Data/img_recommendMe/522627_the_gentlemen.jpg,2026-01-29 15:07:32
1,2,593402,0,/8TiYxxck4kfKwXAAdi6aZLeQd5L.jpg,it,Checco is a young Apulian entrepreneur dreamer...,"['checco', 'apulian entrepreneur', 'sushi rest...",7.025,/CqUxog8F6aaK97RYh8YXhv3NDL.jpg,2020-01-01,Tolo Tolo,Checco Zalone,"['Checco Zalone', 'Manda Touré', 'Nassor Said ...",6.111,1296,Data/img_recommendMe/593402_tolo_tolo.jpg,2026-01-29 15:07:35


In [5]:
batman_movie = movie_df.iloc[110]
batman_movie

id                                                                 111
id_tmdb                                                         576560
adult                                                                0
backdrop_path                         /do2FxGgBYKO1hsF8sA1NmIBvs3m.jpg
original_lenguaje                                                   en
overview             Pushed to his breaking point, a master welder ...
keywords             ['fortifies bulldozer', 'welder', 'seeks destr...
popularity                                                       1.482
poster_path                           /sXh2U3Pj2iMjzzW993kUvHE10RK.jpg
release_date                                                2020-02-21
title                                                            Tread
director                                                    Paul Solet
actors               ['Robert Fleet', 'Mike McCann', 'Kelly Ryan', ...
vote_average                                                     6.600
vote_c

## Get all vectors individualy

### Get adult vector

In [6]:
def multiply_num_vector(num, dimensions=2):
    vector = []

    for i in range(dimensions):
        vector.append(float(num))

    return vector

In [7]:
multiply_num_vector(batman_movie['adult'],2)

[0.0, 0.0]

### Get original lenguaje vector

In [8]:
def hashing_trick(text, dimensions=64, seed_offset=0):

    if text == "" or text == None or text == []:
        return np.zeros(dimensions, dtype=np.float32)

    if not text or not isinstance(text, str):
        text = str(text)
    
    hash_num = mmh3.hash(text, seed=42)
    
    vector = np.zeros(dimensions, dtype=np.float32)
    
    prime1 = 2654435761  
    prime2 = 5915587277
    
    for i in range(dimensions):

        seed = (hash_num ^ (i * prime1) ^ (seed_offset * prime2)) & 0xFFFFFFFF
        
        rng = np.random.RandomState(seed)

        value = rng.uniform(low=-10.0, high=10.0)
        vector[i] = value
    
    return vector

In [9]:
hashing_trick(batman_movie['original_lenguaje'],4)

array([ 3.4582613, -8.665419 , -4.83918  ,  9.013555 ], dtype=float32)

### Get title vector

In [10]:
def hashing_trick_list(list_text, dimensions=64, seed_offset=0):

    if list_text == "" or list_text == None or list_text == []:
        return np.zeros(dimensions, dtype=np.float32)

    text_vectors = []
    for text in list_text:
        if not text or not isinstance(text, str):
            text = str(text)
        
        hash_num = mmh3.hash(text, seed=42)
        
        vector = np.zeros(dimensions, dtype=np.float32)
        
        prime1 = 2654435761  
        prime2 = 5915587277
        
        for i in range(dimensions):
    
            seed = (hash_num ^ (i * prime1) ^ (seed_offset * prime2)) & 0xFFFFFFFF
            
            rng = np.random.RandomState(seed)
    
            value = rng.uniform(low=-10.0, high=10.0)
            vector[i] = value
            text_vectors.append(vector)
    
    return np.mean(text_vectors, axis=0)

In [11]:
list_title = list(batman_movie['title'])
hashing_trick_list(list_title,3)

array([-0.09674486,  0.4463041 , -3.394736  ], dtype=float32)

### Get keywords vector

In [12]:
model = api.load("glove-wiki-gigaword-50")

In [13]:
def top_words_5_vector(list_text, reducir_a_5D=True):

    if list_text == "" or list_text == None or list_text == []:
        return np.zeros(5, dtype=np.float32)

    top_5_vectors = []
    for text in list_text:
    
        words = text.lower().split()
        words_vectors = []
    
        for word in words:
            try:
                vector_50d = model[word]
            except KeyError:
                vector_50d = np.zeros(50)
            
            if not reducir_a_5D:
                return vector_50d
            
            vector_5d = np.zeros(5)
            bloque = 50 // 5 
    
            for i in range(5):
                inicio = i * bloque
                fin = inicio + bloque
                vector_5d[i] = np.mean(vector_50d[inicio:fin])
                
            words_vectors.append(vector_5d)
        top_5_vectors.append(np.mean(words_vectors, axis=0))
        
    return np.round(np.mean(top_5_vectors, axis=0), 7)

In [14]:
keywords_list = ast.literal_eval(batman_movie['keywords'])
top_words_5_vector(keywords_list)

array([ 0.0737301,  0.0821995, -0.0836377,  0.0037927, -0.0998564])

### Get popularity vector

In [15]:
print(multiply_num_vector(batman_movie['popularity'], 3))

[1.482, 1.482, 1.482]


### Get date vector

In [16]:
def date_vector(release_date, num=3):
    year = str(release_date).split('-')[0]
    year = int(year) - 2000
    return multiply_num_vector(year, num)

In [17]:
date_vector(batman_movie['release_date'])

[20.0, 20.0, 20.0]

### Get director vector

In [18]:
hashing_trick(batman_movie['director'] ,5)

array([7.68838  , 2.8533568, 5.6163015, 4.477386 , 1.2732053],
      dtype=float32)

### Get actors vector

In [19]:
actors_list = ast.literal_eval(batman_movie['actors'])
hashing_trick_list(actors_list,5)

array([-0.95966446,  2.394908  , -0.9665145 , -2.945136  , -1.0375808 ],
      dtype=float32)

### Get vote_average vector

In [20]:
multiply_num_vector(batman_movie['vote_average'], 3)

[6.6, 6.6, 6.6]

### Get vote_count

In [21]:
vote_count = float(batman_movie['vote_count']) / 1000
multiply_num_vector(vote_count, 3)

[0.064, 0.064, 0.064]

### Get gender vector

In [22]:
def get_gender_movie(movie_id, dimension=7):

    conexion = mysql.connector.connect(**config)
    cursor = conexion.cursor()
    consulta = "SELECT genre_id FROM Movie_Genres Where movie_id = " + str(movie_id)
    cursor.execute(consulta)
    
    results = cursor.fetchall()
    gender = [x[0] for x in results]
    if gender == "" or gender == None:
        return np.zeros(dimension, dtype=np.float32)
    return hashing_trick_list(gender,dimension)

In [23]:
get_gender_movie(batman_movie["id"])

array([ 2.449926  ,  3.220671  , -6.8632817 ,  4.2147765 ,  1.019425  ,
       -0.12754276,  2.0236647 ], dtype=float32)

### Generate new Dataframe Vectorized

In [24]:
def safe_literal_eval(x):
    if pd.isna(x):
        return []
    
    if isinstance(x, list):
        return x  # Ya es lista
    
    if isinstance(x, str):
        x = x.strip()
        if x and x.startswith('[') and x.endswith(']'):
            try:
                return ast.literal_eval(x)
            except:
                pass
        return [x] if x else []
    
    return []

In [25]:
df_vectorized = pd.DataFrame({
    "id": movie_df['id'],
    'adult': movie_df['adult'].apply(lambda x: multiply_num_vector(x, 2)),
    'original_lenguaje': movie_df['original_lenguaje'].apply(lambda x: hashing_trick(x, 4)),
    'title': movie_df['title'].apply(lambda x: hashing_trick_list(x, 3)),
    'keywords': movie_df['keywords'].apply(lambda x: top_words_5_vector(ast.literal_eval(x))),
    'popularity': movie_df['popularity'].apply(lambda x: multiply_num_vector(x, 3)),
    'release_date': movie_df['release_date'].apply(lambda x: date_vector(x, 3)),
    'director': movie_df['director'].apply(lambda x: hashing_trick(x, 5)),
    'actors': movie_df['actors'].apply(lambda x: hashing_trick_list(safe_literal_eval(x), 5)),
    'vote_average': movie_df['vote_average'].apply(lambda x: multiply_num_vector(x, 3)),
    'vote_count': movie_df['vote_count'].apply(lambda x: multiply_num_vector(float(x) / 1000, 3)),
    'gender': movie_df['id'].apply(lambda x: get_gender_movie(x,7))
})

vector_columns = ['adult','original_lenguaje','title','keywords','popularity',
                 'release_date', 'director', 'actors', 'vote_average',
                 'vote_count', 'gender']
df_vectorized['movie_vector'] = df_vectorized[vector_columns].apply(
    lambda row: np.concatenate([row[col] for col in vector_columns]),
    axis=1
)

In [26]:
df_vectorized.head()

,id,adult,original_lenguaje,title,keywords,popularity,release_date,director,actors,vote_average,vote_count,gender,movie_vector
0,1,"[0.0, 0.0]","[3.4582613, -8.665419, -4.83918, 9.013555]","[2.6661797, -3.3864875, 0.39581978]","[0.0292304, 0.1380805, -0.0593334, 0.0849153, ...","[8.102, 8.102, 8.102]","[20.0, 20.0, 20.0]","[4.1047816, 0.08581555, 6.752872, 7.0444393, -...","[-5.056578, -2.3772938, -0.25809345, -4.423000...","[7.67, 7.67, 7.67]","[6.379, 6.379, 6.379]","[6.4743495, -1.3853433, -2.945437, 1.1297405, ...","[0.0, 0.0, 3.458261251449585, -8.6654186248779..."
1,2,"[0.0, 0.0]","[-5.786882, 3.1440923, 7.295141, -1.9689447]","[-1.8971353, -3.4220293, -3.4715545]","[-0.0535207, 0.1102244, -0.0523355, 0.1846962,...","[7.025, 7.025, 7.025]","[20.0, 20.0, 20.0]","[0.1136715, -7.152463, -3.8764446, -9.506219, ...","[3.5801911, -2.1599047, 4.4701686, 1.308789, -...","[6.111, 6.111, 6.111]","[1.296, 1.296, 1.296]","[-0.59960306, 0.610429, -3.401005, 2.1130917, ...","[0.0, 0.0, -5.786881923675537, 3.1440923213958..."
2,3,"[0.0, 0.0]","[1.0170418, 5.949505, 3.962075, 5.0450106]","[1.6335598, -7.2679763, -5.1170783]","[0.0379357, 0.1357496, 0.023726, 0.0681943, -0...","[2.741, 2.741, 2.741]","[20.0, 20.0, 20.0]","[0.13534275, 7.8766494, 4.26421, 7.2233577, 6....","[-0.56010157, 0.20770058, 1.0595436, -2.607493...","[6.407, 6.407, 6.407]","[0.606, 0.606, 0.606]","[3.686291, -3.8378987, -2.0734096, -6.8887925,...","[0.0, 0.0, 1.017041802406311, 5.94950485229492..."
3,4,"[0.0, 0.0]","[3.4582613, -8.665419, -4.83918, 9.013555]","[-0.7466286, -3.1967628, -1.0123963]","[-0.0454193, 0.1004371, -0.1097798, 0.0915829,...","[7.109, 7.109, 7.109]","[20.0, 20.0, 20.0]","[9.445706, 1.2789574, 0.7857568, 5.3313212, 5....","[0.1782869, -2.6871014, -1.2585841, 2.7482595,...","[6.585, 6.585, 6.585]","[3.812, 3.812, 3.812]","[3.779185, -1.420076, -3.1509016, -1.8540832, ...","[0.0, 0.0, 3.458261251449585, -8.6654186248779..."
4,5,"[0.0, 0.0]","[-5.786882, 3.1440923, 7.295141, -1.9689447]","[3.7885103, -1.9176075, -2.2093558]","[0.2024644, 0.0492065, -0.039987, 0.3319864, -...","[4.345, 4.345, 4.345]","[20.0, 20.0, 20.0]","[-6.5550203, -3.2167592, 8.295671, -0.9372274,...","[-0.73840684, 0.3255829, -1.1704512, 1.1841886...","[7.404, 7.404, 7.404]","[0.704, 0.704, 0.704]","[-4.266442, 4.1208625, -5.2810054, 5.5293097, ...","[0.0, 0.0, -5.786881923675537, 3.1440923213958..."


In [27]:
len(df_vectorized['movie_vector'].iloc[0])

43

In [28]:
df_vectorized['movie_vector'].iloc[0]

array([ 0.        ,  0.        ,  3.45826125, -8.66541862, -4.83917999,
        9.01355457,  2.66617966, -3.38648748,  0.39581978,  0.0292304 ,
        0.1380805 , -0.0593334 ,  0.0849153 ,  0.0385541 ,  8.102     ,
        8.102     ,  8.102     , 20.        , 20.        , 20.        ,
        4.10478163,  0.08581555,  6.75287199,  7.04443932, -0.74707657,
       -5.05657816, -2.37729383, -0.25809345, -4.42300034, -2.0117805 ,
        7.67      ,  7.67      ,  7.67      ,  6.379     ,  6.379     ,
        6.379     ,  6.4743495 , -1.38534331, -2.94543695,  1.12974048,
        4.61370134,  0.68000448,  4.62146616])

In [29]:
df_vectorized.shape

(4254, 13)

#### Variables a conciderar
- adult (2)
- original_lenguaje (4)
- title (3)
- keywords (5)
- popularity (3)
- date (year) (3)
- director (5)
- actors (5)
- vote_average (3)
- vote_count (3)
- Gender (7)
-------------------------
43

## Validate distance between movies

In [37]:
def crear_indice_annoy(vectores, n_arboles=10):
    """Crea índice Annoy"""
    dimension = vectores.shape[1]
    idx = AnnoyIndex(dimension, 'angular')  # 'angular' = similitud coseno
    
    for i, vec in enumerate(vectores):
        idx.add_item(i, vec)
    
    idx.build(n_arboles)
    return idx

def buscar_similares_annoy(indice, vector, k=5):
    """Busca k vecinos más cercanos"""
    return indice.get_nns_by_vector(vector, k, include_distances=True)

In [38]:
indice = crear_indice_annoy(np.vstack(df_vectorized['movie_vector'].values))

In [39]:
batman_vector_test = df_vectorized.iloc[110]['movie_vector']
buscar_similares_annoy(indice, batman_vector_test,5)

([110, 3718, 3295, 1309, 1949],
 [0.000295598671073094,
  0.2637985050678253,
  0.3083724081516266,
  0.31121954321861267,
  0.31370118260383606])

In [40]:
movie_df.iloc[143]

id                                                                 144
id_tmdb                                                         502425
adult                                                                0
backdrop_path                         /vwbmp3vvX4U0VjinaQktYOBt5kW.jpg
original_lenguaje                                                   en
overview             South Africa, 1978. Tim Jenkin and Stephen Lee...
keywords             ['imprisoned apartheid', 'lee white', 'tim jen...
popularity                                                       1.980
poster_path                           /8GGS0jkFFCnmdStvZED6NL6V7gd.jpg
release_date                                                2020-03-06
title                                             Escape from Pretoria
director                                                 Francis Annan
actors               ['Daniel Radcliffe', 'Daniel Webber', 'Ian Har...
vote_average                                                     7.180
vote_c

## Make Test

In [41]:
indice = crear_indice_annoy(np.vstack(df_vectorized['movie_vector'].values))

In [42]:
test_movie = movie_df[movie_df['title'].str.contains('Sonic')]
test_movie

,id,id_tmdb,adult,backdrop_path,original_lenguaje,overview,keywords,popularity,poster_path,release_date,title,director,actors,vote_average,vote_count,image_path,created_at
85,86,454626,0,/stmYfCUGd8Iy6kAMBr6AmWqx8Bq.jpg,en,"Powered with incredible speed, Sonic The Hedge...","['sonic hedgehog', 'stop robotnik', 'vs super'...",10.221,/aQvJ5WPzZgYVDrxLX4R6cLJCEaQ.jpg,2020-02-12,Sonic the Hedgehog,Jeff Fowler,"['Ben Schwartz', 'James Marsden', 'Tika Sumpte...",7.292,10226,Data/img_recommendMe/454626_sonic_the_hedgehog...,2026-01-29 15:10:02
1647,1648,675353,0,/xuLA0pii2IMJW2puT7EvJtgpg0H.jpg,en,"After settling in Green Hills, Sonic is eager ...","['hills sonic', 'knuckles search', 'robotnik r...",11.295,/6DrHO1jr3qVrViUO6s6kFiAGM7.jpg,2022-03-30,Sonic the Hedgehog 2,Jeff Fowler,"['Ben Schwartz', 'James Marsden', 'Tika Sumpte...",7.442,5747,Data/img_recommendMe/675353_sonic_the_hedgehog...,2026-01-29 15:50:19
3741,3743,939243,0,/noPEm6Vu9Lm6QUdTGLOyRgk4M6s.jpg,en,"Sonic, Knuckles, and Tails reunite against a p...","['sonic knuckles', 'tails reunite', 'adversary...",18.190,/d8Ryb8AunYAuycVKDp5HpdWPKgC.jpg,2024-12-19,Sonic the Hedgehog 3,Jeff Fowler,"['Jim Carrey', 'Ben Schwartz', 'Keanu Reeves',...",7.632,3118,Data/img_recommendMe/939243_sonic_the_hedgehog...,2026-01-29 16:51:07


In [43]:
movie = movie_df.iloc[85]
movie

id                                                                  86
id_tmdb                                                         454626
adult                                                                0
backdrop_path                         /stmYfCUGd8Iy6kAMBr6AmWqx8Bq.jpg
original_lenguaje                                                   en
overview             Powered with incredible speed, Sonic The Hedge...
keywords             ['sonic hedgehog', 'stop robotnik', 'vs super'...
popularity                                                      10.221
poster_path                           /aQvJ5WPzZgYVDrxLX4R6cLJCEaQ.jpg
release_date                                                2020-02-12
title                                               Sonic the Hedgehog
director                                                   Jeff Fowler
actors               ['Ben Schwartz', 'James Marsden', 'Tika Sumpte...
vote_average                                                     7.292
vote_c

In [44]:
test = df_vectorized.iloc[85]['movie_vector']
buscar_similares_annoy(indice, test,5)

([85, 1647, 1433, 2530, 1823],
 [0.00031008184305392206,
  0.231051966547966,
  0.28900814056396484,
  0.32614538073539734,
  0.34970614314079285])

In [48]:
close_movie = movie_df.iloc[1823]
close_movie

id                                                                1824
id_tmdb                                                         616037
adult                                                                0
backdrop_path                         /jsoz1HlxczSuTx0mDl2h0lxy36l.jpg
original_lenguaje                                                   en
overview             After his retirement is interrupted by Gorr th...
keywords             ['thor odinson', 'god butcher', 'valkyrie korg...
popularity                                                      11.506
poster_path                           /pIkRyD18kl4FhoCNQuWxWu5cBLM.jpg
release_date                                                2022-07-06
title                                           Thor: Love and Thunder
director                                                 Taika Waititi
actors               ['Chris Hemsworth', 'Natalie Portman', 'Christ...
vote_average                                                     6.400
vote_c

### Get CSV from Database

In [32]:
conexion = mysql.connector.connect(**config)
cursor = conexion.cursor()
consulta = "SELECT * FROM Movie_Genres"
cursor.execute(consulta)

results = cursor.fetchall()
column_names = [desc[0] for desc in cursor.description]

movie_Genres_df = pd.DataFrame(results, columns=column_names)

In [33]:
movie_df.to_csv("../movies_db.csv", index=False)
movie_Genres_df.to_csv("../movie_Genres_df.csv", index=False)
df_vectorized.to_csv("../vectorized_movies_df.csv", index=False)